<a href="https://colab.research.google.com/github/spdb9876/loan-default-prediction/blob/main/01_data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import zipfile
import pandas as pd

# Unzip the downloaded file
zip_file_name = f"./{file_name}.zip"
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall(output_directory)
print(f"Extracted {file_name} from {zip_file_name} to {output_directory}/")

# List files in the directory to confirm extraction
!ls -lh {output_directory}

NameError: name 'file_name' is not defined

In [ ]:
import os
from google.colab import userdata

# Step 1: Install the Kaggle API client
!pip install kaggle --quiet

# Retrieve the Kaggle API key from Colab secrets
# retrive from enviornment variable
kaggle_json_content = userdata.get('KAGGLE_JSON')

# Create the ~/.kaggle directory if it doesn't exist
os.makedirs('/root/.kaggle', exist_ok=True)

# Write the content to kaggle.json
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_json_content)

# Set correct permissions (important for security)
os.chmod('/root/.kaggle/kaggle.json', 600)

print("Kaggle API key setup complete using Colab Secrets.")

# Step 3: Download the specified file from the competition
competition_name = "home-credit-default-risk"
# Correcting the file name: often Kaggle uses underscores instead of periods.
file_name = "application_train.csv"
output_directory = "."

print(f"Attempting to download {file_name} from {competition_name}...")
!kaggle competitions download -c {competition_name} -f {file_name} -p {output_directory}


print(f"Download command executed. Check the directory '{output_directory}' for the file.")

In [ ]:
# Load the extracted CSV file into a pandas DataFrame
df = pd.read_csv(f"{output_directory}/{file_name}")

# Display the first 5 rows of the DataFrame
print(f"\nDisplaying the first 5 rows of {file_name}:")
display(df.head())



```
# This is formatted as code
```



In [ ]:
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100

print(missing_data)
print("======================")
print(missing_percentage)

# Create a DataFrame for missing values
missing_info = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percentage
})

# Filter out columns with no missing values and sort by percentage
missing_info = missing_info[missing_info['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

print("\nMissing Data Analysis for application_train.csv:")
display(missing_info)

### Strategies for Handling Missing Values

When dealing with missing values, the strategy often depends on the percentage of missing data and the nature of the column (numerical or categorical).

1.  **Drop Columns with High Missing Percentage**: For columns with a very high percentage of missing values (e.g., over 60-70%), it's often best to drop them as imputing them might introduce too much noise or bias. These columns might not provide much predictive power.
    *   **Example from `missing_info`**: `COMMONAREA_MEDI`, `COMMONAREA_MODE`, `COMMONAREA_AVG`, `NONLIVINGAPARTMENTS_MODE`, etc., all have around 70% missing data.

2.  **Imputation for Numerical Columns (Moderate Missing Percentage)**:
    *   **Median/Mean Imputation**: For numerical features with a moderate amount of missing data, imputing with the median (less sensitive to outliers) or mean can be effective.
    *   **Example**: Columns like `AMT_REQ_CREDIT_BUREAU_DAY` (around 13.5% missing) or `EXT_SOURCE_1` (around 56% missing) could be candidates for this, depending on domain knowledge and further analysis.

3.  **Imputation for Categorical Columns (Moderate Missing Percentage)**:
    *   **Mode Imputation**: For categorical features, imputing with the mode (most frequent category) is a common practice.
    *   **Treat as a Separate Category**: Sometimes, the fact that a value is missing can itself be informative. In such cases, you can treat missing values as a new, distinct category.

4.  **Drop Rows (Low Missing Percentage)**: If a column has a very small percentage of missing values (e.g., less than 5%) and dropping these few rows won't significantly reduce your dataset size, it can be a simple and effective strategy, especially if imputation is complex or inappropriate for the feature.
    *   **Example**: `EXT_SOURCE_2` has only 0.21% missing data, making it a candidate for row dropping if `SK_ID_CURR` is the unique identifier for rows and imputation is not preferred.

---

### Applying Missing Value Strategies

Let's apply some of these strategies. We'll start by dropping columns with more than 60% missing values.

In [ ]:
# Get columns with more than 60% missing values
threshold = 60
columns_to_drop = missing_info[missing_info['Missing Percentage'] > threshold].index.tolist()

print(f"Dropping {len(columns_to_drop)} columns with more than {threshold}% missing values:")
print(columns_to_drop)

# Drop these columns from the DataFrame
df_cleaned = df.drop(columns=columns_to_drop)

print(f"DataFrame shape after dropping columns: {df_cleaned.shape}")


Next, let's impute remaining numerical columns with their median and categorical columns with their mode. We'll recalculate missing values for the `df_cleaned` DataFrame first.

In [ ]:
# Recalculate missing info for the cleaned DataFrame
missing_data_cleaned = df_cleaned.isnull().sum()
missing_percentage_cleaned = (missing_data_cleaned / len(df_cleaned)) * 100

missing_info_cleaned = pd.DataFrame({
    'Missing Count': missing_data_cleaned,
    'Missing Percentage': missing_percentage_cleaned
})

missing_info_cleaned = missing_info_cleaned[missing_info_cleaned['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

print("\nMissing Data Analysis for df_cleaned (after dropping high-missing columns):")
display(missing_info_cleaned)

# Identify numerical and categorical columns with missing values for imputation
numerical_cols_with_missing = df_cleaned[missing_info_cleaned.index].select_dtypes(include=['int64', 'float64']).columns
categorical_cols_with_missing = df_cleaned[missing_info_cleaned.index].select_dtypes(include=['object', 'category']).columns

# Impute numerical columns with median
for col in numerical_cols_with_missing:
    if df_cleaned[col].isnull().any():
        median_val = df_cleaned[col].median()
        df_cleaned[col].fillna(median_val, inplace=True)
        print(f"Imputed numerical column '{col}' with median: {median_val}")

# Impute categorical columns with mode
for col in categorical_cols_with_missing:
    if df_cleaned[col].isnull().any():
        mode_val = df_cleaned[col].mode()[0] # .mode() can return multiple values, take the first
        df_cleaned[col].fillna(mode_val, inplace=True)
        print(f"Imputed categorical column '{col}' with mode: {mode_val}")

print("\nMissing value imputation complete. Verifying no missing values remain:")
print(df_cleaned.isnull().sum().sum()) # Should be 0 if all missing values are handled


In [ ]:
print("Total missing values remaining in df_cleaned:", df_cleaned.isnull().sum().sum())

In [ ]:
print("\n--- Outlier Handling ---\n")

# --- Specific Anomaly Handling for DAYS_EMPLOYED ---
# In this dataset, a common anomaly in 'DAYS_EMPLOYED' is the value 365243,
# which typically signifies missing or erroneous age data (e.g., 1000 years old).
# We will replace this specific "abnormally high" value with NaN.
print("Checking for and handling specific 'DAYS_EMPLOYED' anomaly (value = 365243)...")
anomaly_count = df_cleaned[df_cleaned['DAYS_EMPLOYED'] == 365243].shape[0]
if anomaly_count > 0:
    df_cleaned['DAYS_EMPLOYED'] = df_cleaned['DAYS_EMPLOYED'].replace({365243: float('nan')},inplace=True)
    print(f"Replaced {anomaly_count} anomalous 'DAYS_EMPLOYED' values (365243) with NaN.")
else:
    print("No specific 'DAYS_EMPLOYED' anomaly (365243) found.")

df_cleaned['AMT_INCOME_TOTAL'] = df_cleaned['AMT_INCOME_TOTAL'].clip(df['AMT_INCOME_TOTAL'], 0, 1e6)

# --- General Outlier Capping ---
# Select only numerical columns for outlier detection
numerical_cols = df_cleaned.select_dtypes(include=['int64', 'float64']).columns

# Exclude 'SK_ID_CURR' and 'TARGET' as they are identifiers/target variable
numerical_cols = numerical_cols.drop(['SK_ID_CURR', 'TARGET'], errors='ignore')

print(f"\nApplying general outlier capping to {len(numerical_cols)} numerical columns (excluding 'DAYS_BIRTH' if already handled)...")

for col in numerical_cols:
    # We'll skip 'DAYS_BIRTH' if it has NaNs (from previous anomaly handling) or if we want specific treatment for it
    # The previous anomaly handling replaces 365243 with NaN. Now, we might want to impute these NaNs if they exist.
    if col == 'DAYS_EMPLOYED':
        # Re-impute 'DAYS_BIRTH' NaNs with median after anomaly detection
        if df_cleaned['DAYS_EMPLOYED'].isnull().any():
            median_birth = df_cleaned['DAYS_EMPLOYED'].median()
            df_cleaned['DAYS_EMPLOYED'].fillna(median_birth, inplace=True)
            print(f"Imputed NaNs in 'DAYS_EMPLOYED' with median: {median_birth}")
        continue # Skip general capping for DAYS_BIRTH as it was specifically treated

    Q1 = df_cleaned[col].quantile(0.01)
    Q3 = df_cleaned[col].quantile(0.99)

    # Using the IQR method for capping as discussed earlier
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_cleaned[col] = df_cleaned[col].clip(lower=lower_bound, upper=upper_bound)

print("Outlier capping complete for numerical columns.")

# Display some descriptive statistics to show the effect of capping
print("\nDescriptive statistics for a few capped numerical columns (after outlier handling):")
display(df_cleaned[numerical_cols].sample(5).describe())